In [1]:
print("Test")

Test


In [2]:
# Imports
import sys
import pathlib

# Add the project's root directory to the Python path
sys.path.append(pathlib.Path("../").resolve().as_posix())

# Configurations
seed = 42

# Paths
DATA_DIR = pathlib.Path("../data/")
ENC2017_ROOT = DATA_DIR / "enc2017"
UD_ET_EDT_ROOT = DATA_DIR / "ud_et_edt"
HOMONYMS_ROOT = DATA_DIR / "homonymous_word_forms"

ENC2017_DIRS = {
    "processed": ENC2017_ROOT / "processed",
    "raw": ENC2017_ROOT / "raw",
}

UD_ET_EDT_DIRS = {
    "processed": UD_ET_EDT_ROOT / "processed",
    "raw": UD_ET_EDT_ROOT / "raw",
}

HOMONYMS_DIRS = {
    "processed": HOMONYMS_ROOT / "processed",
    "annotations": HOMONYMS_ROOT / "annotations",
}

OUTPUT_DIR = pathlib.Path("../outputs/")

MODEL_DIR = pathlib.Path("../models/")

In [3]:
from scripts.old.bert_morph_tagger_notebook_functions import NotebookFunctions

e:\Git_projects\EstNLTK\simpletransformers\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
import evaluate
import pandas as pd

from scripts.model import (
    load_df,
    prepare_shared_inputs,
    initialize_model,
    run_single_model_predict,
)

In [5]:
poseval = evaluate.load("evaluate-metric/poseval", module_type="metric")


def custom_metrics(preds, labels):

    # Evaluate using poseval
    result = poseval.compute(predictions=preds, references=labels, zero_division=0)

    return result

In [6]:
# Step 2: Define paths and load shared sentence-level inputs

# Test set paths (uncomment the one you want to use)
# test_set_path = ENC2017_DIRS["processed"] / "model_test_data.csv"
# test_set_path = UD_ET_EDT_DIRS["processed"] / "UD_test.csv"
# test_set_path = HOMONYMS_DIRS["processed"] / "homonyms_test.parquet"
test_set_path = HOMONYMS_DIRS["processed"] / "homonyms_overall.parquet"

# Model paths (uncomment the pair you want to use)
# baseline_model_path = MODEL_DIR / "NER_mudel"
baseline_model_path = MODEL_DIR / "NER_mudel_v2"
# baseline_model_path = MODEL_DIR / "NER_mudel_v2_homonym_full"
# baseline_model_path = MODEL_DIR / "NER_mudel_v2_homonym_50"
expert_model_path = MODEL_DIR / "NER_mudel_v2_homonym_full"
# expert_model_path = MODEL_DIR / "NER_mudel_v2_homonym_50"

# Homonym set path (only needed for Mixture of Experts evaluation where the expert model is chosen when a homonymous word is the word being evaluated)
homonym_list_path = HOMONYMS_DIRS["processed"] / "homonymous_words.txt"

# Load the evaluation metric
poseval_metric = evaluate.load("evaluate-metric/poseval", module_type="metric")

# Load the test set and prepare shared sentence-level inputs for both evaluation modes
test_df = load_df(test_set_path, file_format="auto")
sentences = prepare_shared_inputs(
    test_df,
    sent_id_col="sentence_id",
    text_col="words",
    label_col="labels",
)

print(f"Loaded sentences: {len(sentences)}")

Loaded sentences: 7886


In [7]:
unique_labels = NotebookFunctions.get_unique_labels("../outputs/unique_labels_old.json")

In [8]:
# Initialise the model bundle for the baseline model
baseline_bundle = initialize_model(
    str(baseline_model_path),
    unique_labels=unique_labels,
    use_Roberta_tokenizer=False,
    use_fast_tokenizer=False,
    cleanup=True,
)
# Initialise the model bundle for the expert model (only needed for Mixture of Experts evaluation)
expert_bundle = initialize_model(
    str(expert_model_path),
    unique_labels=unique_labels,
    use_Roberta_tokenizer=False,
    use_fast_tokenizer=False,
    cleanup=True,
)

E:\Git_projects/EstNLTK/EstNLTK_model_training\scripts\model\utils.py:99: UserWarning: Using label mapping from model `id2label` in checkpoint.
  warnings.warn("Using label mapping from model `id2label` in checkpoint.")


In [13]:
outputs = run_single_model_predict(
    model_path=None,
    model_bundle=baseline_bundle,
    sentences=sentences,
    batch_size=8,
    max_length=128,
    ignore_placeholders=False,
    placeholder_labels={"-", "NONE"},
    include_all_tokens=False,
    verbose=True,
)

Initializing model for prediction...
Preparing encoded inputs...
Prediction finished for 7886 sentences.


In [14]:
print(outputs.keys())

dict_keys(['sentences', 'rows'])


In [15]:
# Create a dataframe out of first sentence in sentences
print(
    pd.DataFrame(
        {
            "word": sentences[0][0],
            "true_label": sentences[0][1],
            "pred_label": outputs["sentences"][0]["pred_labels"],
        }
    )
)

                 word true_label pred_label
0          Edinburghi          -     sg g_H
1             agulite          -     pl g_S
2                mehe          -     sg g_S
3              Irvine          -     sg n_H
4              Welshi          -     sg g_H
5                  ja          -          J
6             Glasgow          -     sg g_H
7     tööliskirjaniku          -     sg g_S
8                   ,          -          Z
9             Bookeri          -     sg g_H
10             võitja     sg n_S     sg n_S
11              James          -     sg n_H
12            Kelmani          -     sg g_H
13              puhul          -          K
14               võib          -        b_V
15         tõlketõrke          -     sg g_S
16           tekitada          -       da_V
17       keelekasutus          -     sg n_S
18                  -          -          Z
19            inglise          -          G
20            inglise          -          G
21            keelele          -

In [16]:
print(outputs["sentences"][0])
print(len(outputs["sentences"][0]["word_tokens"]))

{'sentence_index': 0, 'words': ['Edinburghi', 'agulite', 'mehe', 'Irvine', 'Welshi', 'ja', 'Glasgow', 'tööliskirjaniku', ',', 'Bookeri', 'võitja', 'James', 'Kelmani', 'puhul', 'võib', 'tõlketõrke', 'tekitada', 'keelekasutus', '-', 'inglise', 'inglise', 'keelele', 'demonstratiivselt', 'vastanduv', 'proletaarne', 'Scots', '.'], 'word_tokens': [['▁Ed', 'in', 'burg', 'hi'], ['▁a', 'gul', 'ite'], ['▁mehe'], ['▁I', 'rv', 'ine'], ['▁W', 'el', 's', 'hi'], ['▁ja'], ['▁G', 'las', 'go', 'w'], ['▁töölis', 'kirja', 'niku'], ['▁,'], ['▁B', 'oo', 'keri'], ['▁võitja'], ['▁James'], ['▁Kel', 'mani'], ['▁puhul'], ['▁võib'], ['▁tõl', 'ket', 'õr', 'ke'], ['▁tekitada'], ['▁keele', 'kasutus'], ['▁-'], ['▁inglise'], ['▁inglise'], ['▁keelele'], ['▁demonst', 'ratiiv', 'selt'], ['▁vastan', 'duv'], ['▁prole', 'ta', 'arne'], ['▁Sc', 'ots'], ['▁.']], 'pred_labels': ['sg g_H', 'pl g_S', 'sg g_S', 'sg n_H', 'sg g_H', 'J', 'sg g_H', 'sg g_S', 'Z', 'sg g_H', 'sg n_S', 'sg n_H', 'sg g_H', 'K', 'b_V', 'sg g_S', 'da_V', '

In [68]:
from scripts.model.bert_morph_tagger import (
    BertMorphTagger,
    _split_sentence_into_smaller_chunks,
)
import estnltk

import importlib
import scripts.model.bert_morph_tagger as bmt_module

importlib.reload(bmt_module)
BertMorphTagger = bmt_module.BertMorphTagger

In [69]:
bmt = BertMorphTagger(model_location=str(baseline_model_path))

In [70]:
test_df_w_texts = pd.read_csv(HOMONYMS_DIRS["processed"] / "homonyms_overall.csv")

In [71]:
sentences_texts = test_df_w_texts["sentence"].tolist()

In [72]:
sentence_0 = sentences_texts[0]
text = estnltk.Text(sentence_0)
text.tag_layer("sentences")
# text.add_layer(bmt.make_layer(text))
bmt.tag(text)

Text(text='Edinburghi agulite mehe Irvine Welshi ja Glasgow tööliskirjaniku, Bookeri võitja James Kelmani puhul võib tõlketõrke tekitada keelekasutus - inglise inglise keelele demonstratiivselt vastanduv proletaarne Scots.')

In [75]:
word_span = str((74, 80))
for annotation in text.bert_morph_tagging:
    annotation_word_span = str((annotation.start, annotation.end))
    if annotation_word_span == word_span:
        bmt_prediction = annotation.form[0]  # Get the first analysis
        # If the form is still a list, take the first element
        print(f"Original prediction for word span {word_span}: {bmt_prediction}")
        if isinstance(bmt_prediction, list):
            bmt_prediction = bmt_prediction[0]
            print(f"First element of the original prediction list: {bmt_prediction}")

Original prediction for word span (74, 80): sg n


In [62]:
display(text["bert_morph_tagging"])

Layer(name='bert_morph_tagging', attributes=('bert_tokens', 'form', 'partofspeech', 'probability'), spans=SL[Span('Edinburghi', [{'bert_tokens': ['▁Ed', 'in', 'burg', 'hi'], 'form': 'sg g', 'partofspeech': 'H', 'probability': 0.99958}]),
Span('agulite', [{'bert_tokens': ['▁a', 'gul', 'ite'], 'form': 'pl g', 'partofspeech': 'S', 'probability': 0.99959}]),
Span('mehe', [{'bert_tokens': ['▁mehe'], 'form': 'sg g', 'partofspeech': 'S', 'probability': 0.99989}]),
Span('Irvine', [{'bert_tokens': ['▁I', 'rv', 'ine'], 'form': 'sg n', 'partofspeech': 'H', 'probability': 0.99975}]),
Span('Welshi', [{'bert_tokens': ['▁W', 'el', 's', 'hi'], 'form': 'sg g', 'partofspeech': 'H', 'probability': 0.99954}]),
Span('ja', [{'bert_tokens': ['▁ja'], 'form': '', 'partofspeech': 'J', 'probability': 0.99993}]),
Span('Glasgow', [{'bert_tokens': ['▁G', 'las', 'go', 'w'], 'form': 'sg g', 'partofspeech': 'H', 'probability': 0.99949}]),
Span('tööliskirjaniku', [{'bert_tokens': ['▁töölis', 'kirja', 'niku'], 'form': 'sg g', 'partofspeech': 'S', 'probability': 0.99988}]),
Span(',', [{'bert_tokens': [','], 'form': '', 'partofspeech': 'Z', 'probability': 0.99997}]),
Span('Bookeri', [{'bert_tokens': ['▁B', 'oo', 'keri'], 'form': 'sg g', 'partofspeech': 'H', 'probability': 0.99962}]),
Span('võitja', [{'bert_tokens': ['▁võitja'], 'form': 'sg n', 'partofspeech': 'S', 'probability': 0.99982}]),
Span('James', [{'bert_tokens': ['▁James'], 'form': 'sg n', 'partofspeech': 'H', 'probability': 0.99978}]),
Span('Kelmani', [{'bert_tokens': ['▁Kel', 'mani'], 'form': 'sg g', 'partofspeech': 'H', 'probability': 0.99953}]),
Span('puhul', [{'bert_tokens': ['▁puhul'], 'form': '', 'partofspeech': 'K', 'probability': 0.99965}]),
Span('võib', [{'bert_tokens': ['▁võib'], 'form': 'b', 'partofspeech': 'V', 'probability': 0.99996}]),
Span('tõlketõrke', [{'bert_tokens': ['▁tõl', 'ket', 'õr', 'ke'], 'form': 'sg g', 'partofspeech': 'S', 'probability': 0.99621}]),
Span('tekitada', [{'bert_tokens': ['▁tekitada'], 'form': 'da', 'partofspeech': 'V', 'probability': 0.99985}]),
Span('keelekasutus', [{'bert_tokens': ['▁keele', 'kasutus'], 'form': 'sg n', 'partofspeech': 'S', 'probability': 0.99987}]),
Span('-', [{'bert_tokens': ['▁-'], 'form': '', 'partofspeech': 'Z', 'probability': 0.99997}]),
Span('inglise', [{'bert_tokens': ['▁inglise'], 'form': '', 'partofspeech': 'G', 'probability': 0.99919}]),
Span('inglise', [{'bert_tokens': ['▁inglise'], 'form': '', 'partofspeech': 'G', 'probability': 0.99926}]),
Span('keelele', [{'bert_tokens': ['▁keelele'], 'form': 'sg all', 'partofspeech': 'S', 'probability': 0.99971}]),
Span('demonstratiivselt', [{'bert_tokens': ['▁demonst', 'ratiiv', 'selt'], 'form': '', 'partofspeech': 'D', 'probability': 0.99991}]),
Span('vastanduv', [{'bert_tokens': ['▁vastan', 'duv'], 'form': 'sg n', 'partofspeech': 'A', 'probability': 0.99978}]),
Span('proletaarne', [{'bert_tokens': ['▁prole', 'ta', 'arne'], 'form': 'sg n', 'partofspeech': 'A', 'probability': 0.99953}]),
Span('Scots', [{'bert_tokens': ['▁Sc', 'ots'], 'form': 'sg n', 'partofspeech': 'H', 'probability': 0.99974}]),
Span('.', [{'bert_tokens': ['.'], 'form': '', 'partofspeech': 'Z', 'probability': 0.99996}])])

In [63]:
# Check the tags of the first sentence of BertMorphTagger and the outputs from run_single_model_predict to see if they match
print("BertMorphTagger words\tNER model words")
for bmt_word, ner_word in zip(
    text["bert_morph_tagging"].text, outputs["sentences"][0]["words"]
):
    print(f"{bmt_word}\t/\t{ner_word}")
print("BertMorphTagger tokens\tNER model tokens")
# Check the tokens of the first sentence of BertMorphTagger and the outputs from run_single_model_predict to see if they match
for bmt_token, ner_token in zip(
    text["bert_morph_tagging"].bert_tokens, outputs["sentences"][0]["word_tokens"]
):
    print(f"{bmt_token}\t/\t{ner_token}")
print("BertMorphTagger tags\tNER model pred labels\tTrue labels")
# Check the tags of the first sentence of BertMorphTagger and the outputs from run_single_model_predict to see if they match
# BertMorphTagger gives a part of speech tag and a form tag, while the NER model gives a single label, so we will print them together to see if they match
for (bmt_pos, bmt_form), ner_label, true_label in zip(
    zip(text["bert_morph_tagging"].partofspeech, text["bert_morph_tagging"].form),
    outputs["sentences"][0]["pred_labels"],
    outputs["sentences"][0]["true_labels"],
):
    print(f"{bmt_form[0]}_{bmt_pos[0]}\t/\t{ner_label}\t/\t{true_label}")

BertMorphTagger words	NER model words
Edinburghi	/	Edinburghi
agulite	/	agulite
mehe	/	mehe
Irvine	/	Irvine
Welshi	/	Welshi
ja	/	ja
Glasgow	/	Glasgow
tööliskirjaniku	/	tööliskirjaniku
,	/	,
Bookeri	/	Bookeri
võitja	/	võitja
James	/	James
Kelmani	/	Kelmani
puhul	/	puhul
võib	/	võib
tõlketõrke	/	tõlketõrke
tekitada	/	tekitada
keelekasutus	/	keelekasutus
-	/	-
inglise	/	inglise
inglise	/	inglise
keelele	/	keelele
demonstratiivselt	/	demonstratiivselt
vastanduv	/	vastanduv
proletaarne	/	proletaarne
Scots	/	Scots
.	/	.
BertMorphTagger tokens	NER model tokens
[['▁Ed', 'in', 'burg', 'hi']]	/	['▁Ed', 'in', 'burg', 'hi']
[['▁a', 'gul', 'ite']]	/	['▁a', 'gul', 'ite']
[['▁mehe']]	/	['▁mehe']
[['▁I', 'rv', 'ine']]	/	['▁I', 'rv', 'ine']
[['▁W', 'el', 's', 'hi']]	/	['▁W', 'el', 's', 'hi']
[['▁ja']]	/	['▁ja']
[['▁G', 'las', 'go', 'w']]	/	['▁G', 'las', 'go', 'w']
[['▁töölis', 'kirja', 'niku']]	/	['▁töölis', 'kirja', 'niku']
[[',']]	/	['▁,']
[['▁B', 'oo', 'keri']]	/	['▁B', 'oo', 'keri']
[['▁võitja']]	/

In [64]:
sentences_layer = text.sentences

morph_layer = estnltk.Layer(
    name=bmt.output_layer,
    attributes=bmt.output_attributes,
    text_object=text,
    parent="words",
    ambiguous=True,
)

In [65]:
for k, sentence in enumerate(sentences_layer):
    sent_start = sentence.start
    sent_text = sentence.enclosing_text
    # Apply batch processing: split larger input sentence into smaller chunks and
    # process chunk by chunk
    sent_chunks, sent_chunk_indexes = _split_sentence_into_smaller_chunks(sent_text)
    for sent_chunk, (chunk_start, chunk_end) in zip(sent_chunks, sent_chunk_indexes):
        print(f"<s>{sent_chunk}</s>")
        # Get predictions for the sentence
        top_n_predictions = bmt._get_bert_morph_tagging_label_predictions(
            sent_chunk, bmt.get_top_n_predictions
        )

        # Collect token level annotations (a label for each token)
        for token_data in top_n_predictions:
            start, end = token_data["token"][0], token_data["token"][1]
            bert_tokens = token_data["token"][2]
            if start is None or end is None or bert_tokens == "▁":
                continue  # Ignore sentence start and end tokens (<s>, </s>) and word start sign (▁)
            all_labels = [pred["label"] for pred in token_data["predictions"]]
            all_probabilities = [
                pred["probability"] for pred in token_data["predictions"]
            ]
            token_span = (
                sent_start + chunk_start + start,
                sent_start + chunk_start + end,
            )
            print(sent_start, chunk_start, start, end)
            print(token_data)
            print(token_span)
            for label, prob in zip(all_labels, all_probabilities):
                annotation = {
                    "bert_tokens": bert_tokens,
                    "morph_labels": label,
                    "probabilities": prob,
                }
                morph_layer.add_annotation(token_span, **annotation)
            # annotation = {'bert_tokens' : bert_tokens, 'morph_label' : predicted_label }
            # token_level_annotations.append([token_span, annotation, single_label])

# Add annotations
# if bmt.token_level:
#     # Return token level annotations
#     morph_layer

# else:
#     # Aggregate tokens back into words/phrases
#     # Use BertTokens2WordsRewriter to convert BERT tokens to words
#     rewriter = BertTokens2WordsRewriter(
#         bert_tokens_layer=bmt.output_layer,
#         input_words_layer=bmt.words_layer,
#         output_attributes=bmt.output_attributes,
#         output_layer=bmt.output_layer,
#         decorator=rewriter_decorator)

# #     # Rewrite to align BERT tokens with words
#     morph_layer = rewriter.make_layer(text_obj, layers={morph_layer.name: morph_layer})

<s>Edinburghi agulite mehe Irvine Welshi ja Glasgow tööliskirjaniku, Bookeri võitja James Kelmani puhul võib tõlketõrke tekitada keelekasutus - inglise inglise keelele demonstratiivselt vastanduv proletaarne Scots.</s>
0 0 0 2
{'token': (0, 2, '▁Ed'), 'predictions': [{'label': ('sg g', 'H'), 'probability': 0.99958}]}
(0, 2)
0 0 2 4
{'token': (2, 4, 'in'), 'predictions': [{'label': ('sg g', 'H'), 'probability': 0.99956}]}
(2, 4)
0 0 4 8
{'token': (4, 8, 'burg'), 'predictions': [{'label': ('sg g', 'H'), 'probability': 0.99952}]}
(4, 8)
0 0 8 10
{'token': (8, 10, 'hi'), 'predictions': [{'label': ('sg g', 'H'), 'probability': 0.99933}]}
(8, 10)
0 0 11 12
{'token': (11, 12, '▁a'), 'predictions': [{'label': ('pl g', 'S'), 'probability': 0.99959}]}
(11, 12)
0 0 12 15
{'token': (12, 15, 'gul'), 'predictions': [{'label': ('pl g', 'S'), 'probability': 0.99948}]}
(12, 15)
0 0 15 18
{'token': (15, 18, 'ite'), 'predictions': [{'label': ('pl g', 'S'), 'probability': 0.99941}]}
(15, 18)
0 0 19 23
{'t

In [66]:
bmt._get_bert_morph_tagging_label_predictions(text.text, 1)

[{'token': (None, None, '<s>'),
  'predictions': [{'label': ('sg g', 'H'), 'probability': 0.9707}]},
 {'token': (0, 2, '▁Ed'),
  'predictions': [{'label': ('sg g', 'H'), 'probability': 0.99958}]},
 {'token': (2, 4, 'in'),
  'predictions': [{'label': ('sg g', 'H'), 'probability': 0.99956}]},
 {'token': (4, 8, 'burg'),
  'predictions': [{'label': ('sg g', 'H'), 'probability': 0.99952}]},
 {'token': (8, 10, 'hi'),
  'predictions': [{'label': ('sg g', 'H'), 'probability': 0.99933}]},
 {'token': (11, 12, '▁a'),
  'predictions': [{'label': ('pl g', 'S'), 'probability': 0.99959}]},
 {'token': (12, 15, 'gul'),
  'predictions': [{'label': ('pl g', 'S'), 'probability': 0.99948}]},
 {'token': (15, 18, 'ite'),
  'predictions': [{'label': ('pl g', 'S'), 'probability': 0.99941}]},
 {'token': (19, 23, '▁mehe'),
  'predictions': [{'label': ('sg g', 'S'), 'probability': 0.99989}]},
 {'token': (24, 25, '▁I'),
  'predictions': [{'label': ('sg n', 'H'), 'probability': 0.99975}]},
 {'token': (25, 27, 'rv')

In [36]:
bmt_token = BertMorphTagger(model_location=str(baseline_model_path), token_level=True)
txt = estnltk.Text(sentence_0)
txt.tag_layer("sentences")
bmt_token.tag(txt)
for ann in txt["bert_morph_tagging"]:
    print(
        ann.text,
        ann["bert_tokens"],
        ann["form"],
        ann["partofspeech"],
        ann["probability"],
    )

Ed ['▁Ed'] [''] ['G'] [0.98994]
in ['in'] [''] ['G'] [0.9881]
bu ['bu'] [''] ['G'] [0.98738]
rg ['rg'] [''] ['G'] [0.98856]
hi ['hi'] [''] ['G'] [0.99088]
 a ['▁a'] ['sg g'] ['S'] [0.5269]
gu ['gu'] ['sg g'] ['S'] [0.55789]
li ['li'] ['pl g'] ['S'] [0.60686]
te ['te'] ['pl g'] ['S'] [0.81914]
 me ['▁me'] ['sg n'] ['S'] [0.99924]
he ['he'] ['sg n'] ['S'] [0.99759]
 I ['▁I'] ['sg n'] ['H'] [0.99974]
rv ['rv'] ['sg n'] ['H'] [0.99971]
ine ['ine'] ['sg n'] ['H'] [0.99973]
 W ['▁W'] ['sg g'] ['H'] [0.99948]
el ['el'] ['sg g'] ['H'] [0.99943]
s ['s'] ['sg g'] ['H'] [0.99943]
hi ['hi'] ['sg g'] ['H'] [0.99798]
 ja ['▁ja'] [''] ['J'] [0.99992]
 G ['▁G'] ['sg g'] ['H'] [0.99965]
las ['las'] ['sg g'] ['H'] [0.99958]
go ['go'] ['sg g'] ['H'] [0.99964]
w ['w'] ['sg g'] ['H'] [0.99954]
 t ['▁t'] ['sg g'] ['S'] [0.99959]
öö ['öö'] ['sg g'] ['S'] [0.99935]
lis ['lis'] ['sg g'] ['S'] [0.99894]
kir ['kir'] ['sg g'] ['S'] [0.99931]
ja ['ja'] ['sg g'] ['S'] [0.99919]
ni ['ni'] ['sg g'] ['S'] [0.99944]
ku

Simpletransformers


In [16]:
# Add the project's root directory to the Python path
import sys
import pathlib
import evaluate
import pandas as pd

sys.path.append(pathlib.Path("../").resolve().as_posix())

from scripts.old.bert_morph_tagger_notebook_functions import NotebookFunctions

In [22]:
unique_labels = NotebookFunctions.get_unique_labels("../outputs/unique_labels_old.json")

In [23]:
model = NotebookFunctions.initialize_model("../models/NER_mudel_v2/", unique_labels)

In [17]:
poseval = evaluate.load("evaluate-metric/poseval", module_type="metric")


def custom_metrics(preds, labels):

    # Evaluate using poseval
    result = poseval.compute(predictions=preds, references=labels, zero_division=0)

    return result

In [25]:
ud_test_df = pd.read_csv("../data/ud_et_edt/processed/UD_test.csv")

In [24]:
# Evaluate the model
result, model_outputs, preds_list = model.eval_model(
    ud_test_df, extra_metrics=custom_metrics
)

Running Evaluation:   0%|          | 0/12 [00:00<?, ?it/s]e:\Git_projects\EstNLTK\simpletransformers\Lib\site-packages\simpletransformers\ner\ner_model.py:1341: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast():
Running Evaluation: 100%|██████████| 12/12 [00:01<00:00,  7.43it/s]


In [25]:
print(f"Evaluation Loss:{result['eval_loss']:.4f}")
# Pirnt the custom metrics as percentages
print(f"Accuracy: \t{result['extra_metrics']['accuracy']:.2%}")
print(f"Precision: \t{result['extra_metrics']['weighted avg']['precision']:.2%}")
print(f"Recall: \t{result['extra_metrics']['weighted avg']['recall']:.2%}")
print(f"F1 Score: \t{result['extra_metrics']['weighted avg']['f1-score']:.2%}")

Evaluation Loss:0.1823
Accuracy: 	97.64%
Precision: 	97.78%
Recall: 	97.64%
F1 Score: 	97.69%


In [22]:
# Group by sentence_id to get full sentences and corresponding labels
sentences_test = (
    ud_test_df.groupby("sentence_id")
    .agg({"words": lambda x: list(x), "labels": lambda x: list(x)})
    .reset_index()
)

In [25]:
# Print first sentence's predictions and labels
print("\nFirst sentence predictions and labels:")
for pred, label in zip(preds_list[0], sentences_test["labels"][0]):
    print(f"Pred: {pred}\tLabel: {label}")


First sentence predictions and labels:
Pred: D	Label: D
Pred: pl p_A	Label: pl p_A
Pred: pl p_S	Label: pl p_S
Pred: Z	Label: Z
Pred: J	Label: J
Pred: sg n_S	Label: sg g_S
Pred: J	Label: J
Pred: sg p_S	Label: sg g_S
Pred: Z	Label: Z
Pred: me_V	Label: me_V
Pred: D	Label: D
Pred: sg el_H	Label: sg el_H
Pred: Z	Label: Z
Pred: Y	Label: Y
Pred: J	Label: J
Pred: sg g_H	Label: sg g_H
Pred: sg n_H	Label: sg n_H
Pred: Z	Label: Z
Pred: Y	Label: Y
Pred: Z	Label: Z
Pred: vad_V	Label: vad_V
Pred: K	Label: K
Pred: O	Label: O
Pred: sg el_S	Label: sg el_S
Pred: sg all_S	Label: sg all_S
Pred: sg g_A	Label: sg g_A
Pred: sg ab_S	Label: sg ab_S
Pred: sg g_S	Label: sg g_S
Pred: Z	Label: Z
Pred: sg n_P	Label: sg n_P
Pred: b_V	Label: b_V
Pred: A	Label: A
Pred: sg g_S	Label: sg g_S
Pred: K	Label: K
Pred: da_V	Label: da_V
Pred: D	Label: D
Pred: sg p_H	Label: sg p_H
Pred: D	Label: D
Pred: sg in_H	Label: sg in_H
Pred: Z	Label: Z
Pred: sg g_H	Label: sg g_H
Pred: sg g_S	Label: sg g_S
Pred: sg n_S	Label: sg n_S
Pre

Torch


In [3]:
# 1) Load libs + helpers
import pandas as pd
import torch
from transformers import DataCollatorForTokenClassification
from seqeval.metrics import classification_report
import sys
import pathlib

# Add the project's root directory to the Python path
sys.path.append(pathlib.Path("../").resolve().as_posix())

# import helper functions from your preprocessing module
from scripts.model.model import (
    initialize_model,
    _group_sentences_from_df,
    prepare_token_classification_data,
    TokenClassificationDataset,
)

e:\Git_projects\EstNLTK\simpletransformers\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [18]:
poseval = evaluate.load("evaluate-metric/poseval", module_type="metric")


def custom_metrics(preds, labels):

    # Evaluate using poseval
    result = poseval.compute(predictions=preds, references=labels, zero_division=0)

    return result

In [4]:
# 2) Read test data and keep required cols
# test_df = pd.read_parquet(
#     "../data/homonymous_word_forms/processed/homonyms_test.parquet"
# )
test_df = pd.read_parquet(
    "../data/homonymous_word_forms/processed/homonyms_overall.parquet"
)
# test_df = pd.read_csv("../data/ud_et_edt/processed/UD_test.csv")
# test_df = pd.read_csv("../data/enc2017/processed/model_test_data.csv")
test_df = test_df[["sentence_id", "words", "labels"]]

In [8]:
unique_labels = NotebookFunctions.get_unique_labels("../outputs/unique_labels_old.json")

In [9]:
# 3) Load model+tokenizer (pass model folder or checkpoint)
model_bundle = initialize_model(
    "../models/NER_mudel_v2/",
    unique_labels=unique_labels,
    use_Roberta_tokenizer=False,
    use_fast_tokenizer=False,
    cleanup=True,
)  # returns model, tokenizer, id2label, label2id
model = model_bundle["model"]
tokenizer = model_bundle["tokenizer"]
id2label = model_bundle["id2label"]
label2id = model_bundle["label2id"]

E:\Git_projects/EstNLTK/EstNLTK_model_training\scripts\model\model.py:119: UserWarning: Using label mapping from model `id2label` in checkpoint.
  warnings.warn("Using label mapping from model `id2label` in checkpoint.")


In [10]:
# 4) Group into sentence-level (list of (words_list, labels_list))
sentences = _group_sentences_from_df(test_df)

In [11]:
# 5) Build encodings & aligned labels (use same max_length you trained with)
encodings, aligned_labels = prepare_token_classification_data(
    tokenizer,
    sentences,
    label2id,
    max_length=128,  # adjust if you used different length
    ignore_placeholders=False,  # choose True if you want to ignore non-target placeholders
)

In [12]:
# 6) Create dataset + dataloader
dataset = TokenClassificationDataset(encodings, aligned_labels)
data_collator = DataCollatorForTokenClassification(tokenizer)
dataloader = torch.utils.data.DataLoader(
    dataset, batch_size=8, collate_fn=data_collator
)

In [13]:
# 7) Eval loop: collect token-level predictions and gold labels (ignore -100)
model.eval()
preds_all = []
labels_all = []
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
with torch.no_grad():
    for batch in dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        logits = outputs.logits.detach().cpu().numpy()
        label_ids = batch["labels"].detach().cpu().numpy()
        pred_ids = logits.argmax(axis=-1)
        for p_row, l_row in zip(pred_ids, label_ids):
            preds = []
            labs = []
            for p, l in zip(p_row, l_row):
                if l == -100:
                    continue
                preds.append(id2label[int(p)])
                labs.append(id2label[int(l)])
            preds_all.append(preds)
            labels_all.append(labs)

In [21]:
# 8) Metrics
results = custom_metrics(preds_all, labels_all)

In [22]:
# Print weighted average metrics as percentages
print(f"Accuracy: \t{results['accuracy']:.2%}")
print(f"Precision: \t{results['weighted avg']['precision']:.2%}")
print(f"Recall: \t{results['weighted avg']['recall']:.2%}")
print(f"F1-score: \t{results['weighted avg']['f1-score']:.2%}")

Accuracy: 	93.22%
Precision: 	94.03%
Recall: 	93.22%
F1-score: 	93.52%


In [23]:
# Load results dataframe
results_df = pd.read_csv(
    "../data/homonymous_word_forms/processed/homonyms_evaluation_results_v2.csv",
    index_col=False,
)

In [24]:
print(results_df.head())

   num  inflection_type  \
0    1                1   
1    1                1   
2    1                1   
3    1                1   
4    1                1   

                                                                                                                                                                                                              sentence  \
0  Edinburghi agulite mehe Irvine Welshi ja Glasgow tööliskirjaniku, Bookeri võitja James Kelmani puhul võib tõlketõrke tekitada keelekasutus - inglise inglise keelele demonstratiivselt vastanduv proletaarne Scots.   
1                                                                               Normi-aktiveerimise teooria (Schwartz, 1970) on algselt mõeldud moraalse otsustamisprotsessi analüüsimiseks abistava käitumise näitel.   
2                                                                                                                "Ehk oleks mõttekas ka mõni selleteemaline hoiatav kampaania korraldad

In [37]:
print(len(results_df))

7886


In [ ]:
bmt_labels = results_df["bmt_prediction"]

In [26]:
# Group by sentence_id to get full sentences and corresponding labels
sentences_test = (
    test_df.groupby("sentence_id")
    .agg({"words": lambda x: list(x), "labels": lambda x: list(x)})
    .reset_index()
)

In [29]:
sentences_test.iloc[0]

sentence_id                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  0
words          [Edinburghi, agulite, mehe, Irvine, Welshi, ja, Glasgow, tööliskirjaniku, ,, Bookeri, võitja, James, Kelmani, puhul, võib, tõlketõrke, tekitada, keelekasutus, -, inglise, inglise, keelele, demonstratiivselt, vastanduv,

In [36]:
pred_count = 0
for preds in preds_all:
    pred_count += len(preds)
print(f"Total predictions: {pred_count}")
labels_count = 0
for labels in labels_all:
    labels_count += len(labels)
print(f"Total labels: {labels_count}")

Total predictions: 4396
Total labels: 4396


In [31]:
# Print first sentence's predictions and labels
print("\nFirst sentence predictions and labels:")
for pred, label in zip(preds_all[0], sentences_test["labels"][0]):
    print(f"Pred: {pred}\tLabel: {label}")


First sentence predictions and labels:
Pred: sg n_S	Label: -
Pred: sg n_S	Label: -
Pred: sg n_S	Label: -
Pred: sg g_S	Label: -
Pred: sg g_H	Label: -


In [37]:
# Check diffs between simpletransformers preds and custom loop preds
print(
    "\nComparing simpletransformers preds and custom loop preds for first five sentences:"
)
for i in range(5):
    print(f"\nSentence {i + 1}:")
    for st_pred, cl_pred, label in zip(
        preds_list[i], preds_all[i], sentences_test["labels"][i]
    ):
        if st_pred != cl_pred:
            print(f"ST Pred: {st_pred}\tCL Pred: {cl_pred}\tLabel: {label}")
# Count total tokens and mismatches
total_tokens = 0
mismatches = 0
for st_preds, cl_preds, labels in zip(preds_list, preds_all, sentences_test["labels"]):
    for st_pred, cl_pred, label in zip(st_preds, cl_preds, labels):
        total_tokens += 1
        if st_pred != cl_pred:
            mismatches += 1
print(
    f"\nTotal tokens: {total_tokens}, Mismatches: {mismatches}",
    f"({mismatches / total_tokens:.2%} mismatch rate)",
)


Comparing simpletransformers preds and custom loop preds for first five sentences:

Sentence 1:
ST Pred: sg n_S	CL Pred: sg p_S	Label: sg g_S
ST Pred: me_V	CL Pred: ?	Label: me_V
ST Pred: D	CL Pred: sg g_S	Label: D
ST Pred: K	CL Pred: pl in_S	Label: K
ST Pred: sg g_S	CL Pred: sg g_H	Label: sg g_S

Sentence 2:
ST Pred: adt_A	CL Pred: sg g_A	Label: adt_A
ST Pred: sg g_S	CL Pred: pl g_S	Label: sg g_S
ST Pred: pl ad_S	CL Pred: sg ad_S	Label: pl ad_S
ST Pred: J	CL Pred: Y	Label: J
ST Pred: D	CL Pred: sg ad_P	Label: D

Sentence 3:
ST Pred: sg g_S	CL Pred: sg n_S	Label: sg g_S
ST Pred: sg g_A	CL Pred: ?	Label: sg g_A
ST Pred: K	CL Pred: sg p_S	Label: K
ST Pred: sg ab_S	CL Pred: da_V	Label: sg ab_S
ST Pred: sg n_P	CL Pred: sg g_P	Label: sg n_P
ST Pred: sg ad_P	CL Pred: sg ad_S	Label: sg ad_P
ST Pred: sg g_P	CL Pred: sg g_A	Label: sg g_P
ST Pred: tud_V	CL Pred: ks_V	Label: tud_V
ST Pred: sg tr_S	CL Pred: ks_V	Label: sg tr_S
ST Pred: sg p_S	CL Pred: sg el_S	Label: sg p_S

Sentence 4:
ST Pred: s

In [37]:
# Check diffs between simpletransformers preds and custom loop preds
print(
    "\nComparing simpletransformers preds and custom loop preds for first five sentences:"
)
for i in range(5):
    print(f"\nSentence {i + 1}:")
    for st_pred, cl_pred, label in zip(
        preds_list[i], preds_all[i], sentences_test["labels"][i]
    ):
        if st_pred != cl_pred:
            print(f"ST Pred: {st_pred}\tCL Pred: {cl_pred}\tLabel: {label}")
# Count total tokens and mismatches
total_tokens = 0
mismatches = 0
for st_preds, cl_preds, labels in zip(preds_list, preds_all, sentences_test["labels"]):
    for st_pred, cl_pred, label in zip(st_preds, cl_preds, labels):
        total_tokens += 1
        if st_pred != cl_pred:
            mismatches += 1
print(
    f"\nTotal tokens: {total_tokens}, Mismatches: {mismatches}",
    f"({mismatches / total_tokens:.2%} mismatch rate)",
)


Comparing simpletransformers preds and custom loop preds for first five sentences:

Sentence 1:
ST Pred: me_V	CL Pred: ?	Label: me_V
ST Pred: D	CL Pred: sg g_S	Label: D
ST Pred: K	CL Pred: pl in_S	Label: K
ST Pred: sg g_S	CL Pred: sg g_H	Label: sg g_S
ST Pred: sg n_S	CL Pred: pl n_S	Label: sg n_S

Sentence 2:
ST Pred: adt_A	CL Pred: sg g_A	Label: adt_A
ST Pred: sg g_S	CL Pred: pl g_S	Label: sg g_S
ST Pred: pl ad_S	CL Pred: sg ad_S	Label: pl ad_S
ST Pred: J	CL Pred: Y	Label: J
ST Pred: D	CL Pred: sg ad_P	Label: D

Sentence 3:
ST Pred: sg g_S	CL Pred: sg n_S	Label: sg g_S
ST Pred: sg g_A	CL Pred: ?	Label: sg g_A
ST Pred: K	CL Pred: sg p_S	Label: K
ST Pred: sg ab_S	CL Pred: da_V	Label: sg ab_S
ST Pred: sg n_P	CL Pred: sg g_P	Label: sg n_P
ST Pred: sg ad_P	CL Pred: sg ad_S	Label: sg ad_P
ST Pred: sg g_P	CL Pred: sg g_A	Label: sg g_P
ST Pred: tud_V	CL Pred: ks_V	Label: tud_V
ST Pred: sg tr_S	CL Pred: ks_V	Label: sg tr_S
ST Pred: sg p_S	CL Pred: sg el_S	Label: sg p_S
ST Pred: b_V	CL Pred: s

Torch


In [8]:
import pandas as pd
import evaluate

In [9]:
ud_test_df = pd.read_csv("../data/ud_et_edt/processed/UD_test.csv")

In [10]:
poseval = evaluate.load("evaluate-metric/poseval", module_type="metric")


def custom_metrics(preds, labels):

    # Evaluate using poseval
    result = poseval.compute(predictions=preds, references=labels)

    return result

In [11]:
# 1) Load libs + helpers
import pandas as pd
import torch
from transformers import DataCollatorForTokenClassification
from seqeval.metrics import classification_report
import sys
import pathlib

# Add the project's root directory to the Python path
sys.path.append(pathlib.Path("../").resolve().as_posix())

# import helper functions from your preprocessing module
from scripts.model.model import (
    initialize_model,
    _group_sentences_from_df,
    prepare_token_classification_data,
    TokenClassificationDataset,
)

In [12]:
# 2) Read test data and keep required cols
test_df = pd.read_csv("../data/ud_et_edt/processed/UD_test.csv")
test_df = test_df[["sentence_id", "words", "labels"]]

In [13]:
# 3) Load model+tokenizer (pass model folder or checkpoint)
model_bundle = initialize_model(
    "../models/NER_mudel_v2/",
    use_fast_tokenizer=False,  # force SimpleTransformers-compatible tokenisation
    use_Roberta_tokenizer=False,
)  # returns model, tokenizer, id2label, label2id
model = model_bundle["model"]
tokenizer = model_bundle["tokenizer"]
id2label = model_bundle["id2label"]
label2id = model_bundle["label2id"]

E:\Git_projects/EstNLTK/EstNLTK_model_training\scripts\model\model.py:116: UserWarning: Using label mapping from model `id2label` in checkpoint.
  warnings.warn("Using label mapping from model `id2label` in checkpoint.")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [14]:
tokenizer = CamembertTokenizer.from_pretrained(
    "../models/NER_mudel_v2/", use_fast=False, backend="sentencepiece"
)

In [15]:
# 4) Group into sentence-level (list of (words_list, labels_list))
sentences = _group_sentences_from_df(test_df)

In [16]:
# 5) Build encodings & aligned labels (use same max_length you trained with)
encodings, aligned_labels = prepare_token_classification_data(
    tokenizer,
    sentences,
    label2id,
    max_length=None,  # adjust if you used different length
    ignore_placeholders=False,  # choose True if you want to ignore non-target placeholders
)

In [17]:
for i, (sent, label) in enumerate(sentences):
    if len(sent) < 30:
        print(i)
        print(sent)
        break

240
['Kuid', 'siis', 'nad', 'tulid', ',', 'elusignaalid', '.', 'Lindistus', '!', 'Kas', 'sari', 'on', 'tänaseks', 'põhjalikult', 'läbi', 'võetud', '?', 'Raha', 'on', 'hea', 'asi', '.', 'Järgnevalt', 'käsitletaksegi', 'nimetatud', 'teguriterühmi', 'lähemalt', '.']


In [20]:
# confirm fast tokenizer
print("is_fast:", tokenizer.is_fast)

# tokenise single sentence and inspect BatchEncoding
enc = tokenizer(sentences[i][0], is_split_into_words=True)
print(
    "tokens:",
    tokenizer.convert_ids_to_tokens(enc["input_ids"]),
    "length:",
    len(enc["input_ids"]),
)

print("input_ids:", enc["input_ids"], "length:", len(enc["input_ids"]))

print("attention_mask:", enc["attention_mask"], "length:", len(enc["attention_mask"]))

print("labels:", sentences[i][1], "length:", len(sentences[i][1]))

print("Aligned labels:", aligned_labels[i], "length:", len(aligned_labels[i]))

is_fast: False
tokens: ['<s>', '▁Kuid', '▁siis', '▁nad', '▁tulid', '▁,', '▁elus', 'ig', 'naal', 'id', '▁.', '▁Lind', 'istus', '▁!', '▁Kas', '▁sari', '▁on', '▁tänaseks', '▁põhjalikult', '▁läbi', '▁võetud', '▁?', '▁Raha', '▁on', '▁hea', '▁asi', '▁.', '▁Järgnevalt', '▁käsitletakse', 'gi', '▁nimetatud', '▁tegu', 'ri', 'ter', 'üh', 'mi', '▁lähemalt', '▁.', '</s>'] length: 39
input_ids: [5, 1113, 160, 549, 5038, 36, 2593, 173, 8649, 20, 42, 10015, 2747, 513, 651, 18991, 45, 8784, 9264, 616, 2979, 414, 7214, 45, 635, 1149, 42, 25987, 20624, 63, 2500, 1118, 41, 452, 464, 250, 8063, 42, 6] length: 39
attention_mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1] length: 39
labels: ['J', 'D', 'pl n_P', 'sid_V', 'Z', 'pl n_S', 'Z', 'sg n_S', 'Z', 'D', 'sg n_S', 'b_V', 'sg tr_A', 'D', 'D', 'tud_V', 'Z', 'sg n_S', 'b_V', 'sg n_A', 'sg n_S', 'Z', 'D', 'takse_V', 'A', 'pl p_S', 'D', 'Z'] length: 28
Aligned labels: [-100, 6, 2, 197

In [18]:
# confirm fast tokenizer
print("is_fast:", tokenizer.is_fast)

# tokenise single sentence and inspect BatchEncoding
enc = tokenizer(sentences[i][0], is_split_into_words=True, return_offsets_mapping=True)
# print(enc)  # full encoding
print(
    "tokens:", enc.tokens(), "length:", len(enc.tokens())
)  # token strings (includes special tokens)
print(
    "input_ids:", enc["input_ids"], "length:", len(enc["input_ids"])
)  # token ids (includes special tokens)
print(
    "attention_mask:", enc["attention_mask"], "length:", len(enc["attention_mask"])
)  # attention mask (includes special tokens)
print(
    "word_ids:", enc.word_ids(), "length:", len(enc.word_ids())
)  # word ids for each token (None for special tokens)
print(
    "offsets:", enc["offset_mapping"], "length:", len(enc["offset_mapping"])
)  # character offsets for each token (includes special tokens)
print(
    "labels:", sentences[i][1], "length:", len(sentences[i][1])
)  # original labels for the sentence
print(
    "labels input format:", aligned_labels[i], "length:", len(aligned_labels[i])
)  # labels aligned to tokens (with -100 for ignored tokens)

is_fast: True
tokens: ['<s>', '▁Ku', 'id', '▁s', 'iis', '▁n', 'ad', '▁t', 'ul', 'id', '▁,', '▁el', 'us', 'ig', 'na', 'al', 'id', '▁.', '▁L', 'in', 'di', 'st', 'us', '▁!', '▁K', 'as', '▁sa', 'ri', '▁on', '▁tä', 'na', 'se', 'ks', '▁põh', 'ja', 'li', 'ku', 'lt', '▁lä', 'bi', '▁võ', 'et', 'ud', '▁?', '▁Ra', 'ha', '▁on', '▁h', 'ea', '▁a', 'si', '▁.', '▁Järg', 'ne', 'va', 'lt', '▁kä', 'sit', 'le', 'ta', 'kse', 'gi', '▁n', 'im', 'et', 'at', 'ud', '▁te', 'gu', 'ri', 'ter', 'üh', 'mi', '▁lä', 'he', 'ma', 'lt', '▁.', '</s>'] length: 79
input_ids: [5, 222, 20, 21, 125, 32, 92, 11, 48, 20, 36, 313, 30, 173, 58, 12, 20, 42, 137, 13, 141, 9, 30, 513, 65, 67, 163, 41, 45, 145, 58, 7, 34, 319, 17, 91, 110, 124, 214, 243, 72, 94, 26, 414, 600, 111, 45, 61, 368, 27, 71, 42, 3479, 81, 35, 124, 205, 895, 10, 25, 178, 63, 32, 59, 94, 82, 26, 50, 85, 41, 452, 464, 250, 214, 107, 22, 124, 42, 6] length: 79
attention_mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,

In [ ]:
from transformers import AutoTokenizer, CamembertTokenizer

path = "../models/NER_mudel_v2/"

tok_a = CamembertTokenizer.from_pretrained(path, use_fast=False)
tok_b = AutoTokenizer.from_pretrained(
    path, backend="sentencepiece"
)  # v5-style explicit backend

for name, tok in [
    ("CamembertTokenizer", tok_a),
    ("AutoTokenizer+sentencepiece", tok_b),
]:
    print(name, type(tok).__name__, "is_fast=", getattr(tok, "is_fast", None))
    print("tokenizer_file:", tok.init_kwargs.get("tokenizer_file"))
    print("vocab_file:", tok.init_kwargs.get("vocab_file"))

e:\Git_projects\EstNLTK\simpletransformers\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CamembertTokenizer CamembertTokenizer is_fast= False
tokenizer_file: ../models/NER_mudel_v2/tokenizer.json
vocab_file: None
AutoTokenizer+sentencepiece CamembertTokenizerFast is_fast= True
tokenizer_file: None
vocab_file: None


In [7]:
import pandas as pd
import sys
import pathlib

# Add the project's root directory to the Python path
sys.path.append(pathlib.Path("../").resolve().as_posix())

from scripts.preprocessing.common import sampler


def test_greedy_pack_exact_fit():
    items = [
        sampler.Item(id="a", size=100, meta={}),
        sampler.Item(id="b", size=200, meta={}),
        sampler.Item(id="c", size=300, meta={}),
    ]
    selected, acc, rounds = sampler.greedy_pack(items, target=300, seed=42)
    assert acc >= 300
    # exact-fit available: should pick the 300-sized item only
    assert len(selected) == 1
    assert selected[0].size == 300
    assert rounds == 0


def test_greedy_pack_multiple_items():
    items = [
        sampler.Item(id=str(i), size=s, meta={})
        for i, s in enumerate([50, 80, 120, 40])
    ]
    selected, acc, rounds = sampler.greedy_pack(items, target=200, seed=1)
    assert acc >= 200
    assert rounds >= 0
    # ensure we selected at least two items in this small pool
    assert len(selected) >= 2


def make_word_df(records):
    """Helper to construct a word-level DataFrame.

    records: list of tuples (source, sentence_id, word_text)
    """
    df = pd.DataFrame(records, columns=["source", "sentence_id", "words"])
    return df


def test_sample_corpus_sentence_mode():
    # Build a small corpus with two sources, sentences with varying word counts
    rows = []
    # source s1: sentence 0 has 3 words, sentence 1 has 2 words
    rows += [("s1", 0, "w1"), ("s1", 0, "w2"), ("s1", 0, "w3")]
    rows += [("s1", 1, "w1"), ("s1", 1, "w2")]
    # source s2: sentence 0 has 4 words
    rows += [("s2", 0, "a"), ("s2", 0, "b"), ("s2", 0, "c"), ("s2", 0, "d")]

    df = make_word_df(rows)

    # target = 4 words -> sampler should pick either s2 sentence(4) or combine s1 sentences
    sampled, prov = sampler.sample_corpus(df, mode="sentence", target_words=4, seed=0)
    # sampled rows count equals achieved words (one row per word)
    assert len(sampled) == prov["achieved_words"]
    assert prov["achieved_words"] >= 4


def test_sample_corpus_source_mode():
    # Build a small corpus where sources have different total sizes
    rows = []
    # source s1: total 5 words
    rows += [
        ("s1", 0, "x"),
        ("s1", 0, "x"),
        ("s1", 1, "x"),
        ("s1", 1, "x"),
        ("s1", 1, "x"),
    ]
    # source s2: total 3 words
    rows += [("s2", 0, "y"), ("s2", 0, "y"), ("s2", 1, "y")]

    df = make_word_df(rows)
    sampled, prov = sampler.sample_corpus(df, mode="source", target_words=4, seed=2)
    # since sampling by source, sampled rows should be whole sources
    # and len(sampled) should equal achieved words
    assert len(sampled) == prov["achieved_words"]
    assert prov["achieved_words"] >= 4


def test_deduplicate_sentences():
    # two corpora with identical sentence texts
    rows = []
    # corpus A: one sentence with two words
    rows += [("A", "src1", 0, "one"), ("A", "src1", 0, "two")]
    # corpus B: same sentence text duplicated
    rows += [("B", "src2", 0, "one"), ("B", "src2", 0, "two")]

    df = pd.DataFrame(rows, columns=["orig_corpus", "source", "sentence_id", "words"])

    deduped = sampler.deduplicate_sentences(df)
    # one of the two sentence-blocks should be marked duplicated
    assert "duplicated_sentence" in deduped.columns
    dup_counts = deduped["duplicated_sentence"].sum()
    assert dup_counts >= 1


In [8]:
print("Starting tests...")
test_greedy_pack_exact_fit()
test_greedy_pack_multiple_items()
test_sample_corpus_sentence_mode()
test_sample_corpus_source_mode()
test_deduplicate_sentences()
print("All tests passed!")

Starting tests...
All tests passed!


## Tokenisation diagnostics: SimpleTransformers vs custom preprocessing

This section compares feature construction **before model forward pass** for the same sentence:

- `input_ids` (tokenisation + special tokens + truncation/padding)
- aligned `label_ids` (first-subtoken labelling and ignored positions)

Run the next code cell after the earlier setup cells where `tokenizer`, `label2id`, and `sentences` are defined.


In [28]:
import pathlib
import sys
import typing

# Ensure local simpletransformers package in this workspace is importable.
workspace_root = pathlib.Path("..", "..").resolve()
local_st_path = workspace_root / "simpletransformers" / "Lib" / "site-packages"
if local_st_path.exists() and local_st_path.as_posix() not in sys.path:
    sys.path.insert(0, local_st_path.as_posix())

from simpletransformers.ner.ner_utils import InputExample, convert_example_to_feature
from scripts.model.model import prepare_token_classification_data


def _has_extra_sep_token(tokenizer_obj: typing.Any) -> bool:
    """Return True for tokenizers that use an extra separator token.

    This matches SimpleTransformers' MODELS_WITH_EXTRA_SEP_TOKEN behaviour
    (roberta/camembert/xlmroberta/longformer/mpnet) using tokenizer metadata.
    """
    signature = " ".join(
        [
            tokenizer_obj.__class__.__name__.lower(),
            str(getattr(tokenizer_obj, "name_or_path", "")).lower(),
        ]
    )
    return any(
        model_name in signature
        for model_name in ("roberta", "camembert", "xlmroberta", "longformer", "mpnet")
    )


def _to_int_label_map(
    label2id_map: typing.Dict[str, typing.Any],
) -> typing.Dict[str, int]:
    """Convert possibly string-typed label ids into int ids.

    Args:
        label2id_map: Label to id mapping from model/config.

    Returns:
        A dictionary with int label ids.
    """
    return {str(label): int(label_id) for label, label_id in label2id_map.items()}


def build_st_feature_for_sentence(
    words: typing.List[str],
    labels: typing.List[str],
    tokenizer_obj: typing.Any,
    label2id_map: typing.Dict[str, int],
    max_seq_length: int = 128,
    pad_token_label_id: int = -100,
) -> typing.Dict[str, typing.List[int]]:
    """Build one sentence feature exactly like SimpleTransformers does.

    Args:
        words: Tokenised sentence words.
        labels: Gold labels aligned to words.
        tokenizer_obj: Hugging Face tokenizer used by the model.
        label2id_map: Mapping from label string to label id.
        max_seq_length: Fixed sequence length used in evaluation/training.
        pad_token_label_id: Ignore index used for non-first subtokens and padding.

    Returns:
        A dictionary containing input_ids, attention_mask, token_type_ids, labels.
    """
    label_map = _to_int_label_map(label2id_map)

    missing_labels = sorted(
        {str(label) for label in labels if str(label) not in label_map}
    )
    if missing_labels:
        raise KeyError(
            "Label(s) from sentence not found in label map: "
            f"{missing_labels[:10]}{' ...' if len(missing_labels) > 10 else ''}"
        )

    example = InputExample(guid="cmp-0", words=words, labels=labels)
    feature = convert_example_to_feature(
        example=example,
        label_map=label_map,
        max_seq_length=max_seq_length,
        tokenizer=tokenizer_obj,
        cls_token_at_end=False,
        cls_token=tokenizer_obj.cls_token,
        cls_token_segment_id=0,
        sep_token=tokenizer_obj.sep_token,
        sep_token_extra=_has_extra_sep_token(tokenizer_obj),
        pad_on_left=False,
        pad_token=tokenizer_obj.pad_token_id,
        pad_token_segment_id=0,
        pad_token_label_id=pad_token_label_id,
        sequence_a_segment_id=0,
        mask_padding_with_zero=True,
    )

    return {
        "input_ids": feature.input_ids,
        "attention_mask": feature.input_mask,
        "token_type_ids": feature.segment_ids,
        "labels": feature.label_ids,
    }


def compare_preprocessing_pipelines(
    sentence_data: typing.List[typing.Tuple[typing.List[str], typing.List[str]]],
    tokenizer_obj: typing.Any,
    label2id_map: typing.Dict[str, int],
    max_seq_length: int = 128,
    sample_count: int = 200,
    print_examples: int = 3,
    pad_token_label_id: int = -100,
) -> None:
    """Compare SimpleTransformers vs custom preprocessing sentence-by-sentence.

    Prints aggregate mismatch statistics and a few concrete mismatch examples.
    """
    checked = min(sample_count, len(sentence_data))
    input_id_mismatches = 0
    label_id_mismatches = 0
    attention_mismatches = 0
    mismatching_sentence_indices: typing.List[int] = []

    for sent_idx in range(checked):
        words, labels = sentence_data[sent_idx]
        st_feature = build_st_feature_for_sentence(
            words=words,
            labels=labels,
            tokenizer_obj=tokenizer_obj,
            label2id_map=label2id_map,
            max_seq_length=max_seq_length,
            pad_token_label_id=pad_token_label_id,
        )

        custom_encodings, custom_aligned_labels = prepare_token_classification_data(
            tokenizer=tokenizer_obj,
            sentences=[(words, labels)],
            label2id=label2id_map,
            max_length=max_seq_length,
            ignore_placeholders=False,
            pad_token_label_id=pad_token_label_id,
        )
        custom_feature = custom_encodings[0]
        custom_label_ids = custom_aligned_labels[0]

        input_diff = st_feature["input_ids"] != custom_feature["input_ids"]
        attn_diff = st_feature["attention_mask"] != custom_feature["attention_mask"]
        label_diff = st_feature["labels"] != custom_label_ids

        if input_diff:
            input_id_mismatches += 1
        if attn_diff:
            attention_mismatches += 1
        if label_diff:
            label_id_mismatches += 1

        if input_diff or attn_diff or label_diff:
            mismatching_sentence_indices.append(sent_idx)

    print("=== Preprocessing comparison summary ===")
    print(f"Sentences checked: {checked}")
    print(
        f"input_ids mismatches: {input_id_mismatches} "
        f"({(input_id_mismatches / checked) if checked else 0.0:.2%})"
    )
    print(
        f"attention_mask mismatches: {attention_mismatches} "
        f"({(attention_mismatches / checked) if checked else 0.0:.2%})"
    )
    print(
        f"label_ids mismatches: {label_id_mismatches} "
        f"({(label_id_mismatches / checked) if checked else 0.0:.2%})"
    )

    if not mismatching_sentence_indices:
        print("No preprocessing differences detected in the inspected sample.")
        return

    print("\n=== Example mismatching sentences ===")
    for sent_idx in mismatching_sentence_indices[:print_examples]:
        words, labels = sentence_data[sent_idx]
        st_feature = build_st_feature_for_sentence(
            words=words,
            labels=labels,
            tokenizer_obj=tokenizer_obj,
            label2id_map=label2id_map,
            max_seq_length=max_seq_length,
            pad_token_label_id=pad_token_label_id,
        )
        custom_encodings, custom_aligned_labels = prepare_token_classification_data(
            tokenizer=tokenizer_obj,
            sentences=[(words, labels)],
            label2id=label2id_map,
            max_length=max_seq_length,
            ignore_placeholders=False,
            pad_token_label_id=pad_token_label_id,
        )
        custom_feature = custom_encodings[0]
        custom_label_ids = custom_aligned_labels[0]

        st_tokens = tokenizer_obj.convert_ids_to_tokens(st_feature["input_ids"])
        custom_tokens = tokenizer_obj.convert_ids_to_tokens(custom_feature["input_ids"])

        print(f"\nSentence index: {sent_idx}")
        print(f"Words: {words[:20]}{' ...' if len(words) > 20 else ''}")

        first_token_diff = next(
            (
                idx
                for idx, (st_tok, custom_tok) in enumerate(
                    zip(st_tokens, custom_tokens)
                )
                if st_tok != custom_tok
            ),
            None,
        )
        first_label_diff = next(
            (
                idx
                for idx, (st_lab, custom_lab) in enumerate(
                    zip(st_feature["labels"], custom_label_ids)
                )
                if st_lab != custom_lab
            ),
            None,
        )

        print(f"First token difference position: {first_token_diff}")
        print(f"First label-id difference position: {first_label_diff}")

        if first_token_diff is not None:
            start = max(0, first_token_diff - 5)
            end = min(len(st_tokens), first_token_diff + 6)
            print("ST tokens window:    ", st_tokens[start:end])
            print("Custom tokens window:", custom_tokens[start:end])

        if first_label_diff is not None:
            start = max(0, first_label_diff - 5)
            end = min(len(st_feature["labels"]), first_label_diff + 6)
            print("ST label ids window:    ", st_feature["labels"][start:end])
            print("Custom label ids window:", custom_label_ids[start:end])


# Run diagnostics on a subset first; increase sample_count as needed.
compare_preprocessing_pipelines(
    sentence_data=sentences,
    tokenizer_obj=tokenizer,
    label2id_map=label2id,
    max_seq_length=128,
    sample_count=2000,
    print_examples=5,
    pad_token_label_id=-100,
)

=== Preprocessing comparison summary ===
Sentences checked: 1198
input_ids mismatches: 0 (0.00%)
attention_mask mismatches: 0 (0.00%)
label_ids mismatches: 0 (0.00%)
No preprocessing differences detected in the inspected sample.


## Next check: eval tensors and model-output parity

Preprocessing parity with one tokenizer is confirmed. The next step compares:

- tensors produced by `simpletransformers` `load_and_cache_examples(...)` vs custom preprocessing
- raw logits/predictions from both models on **the same exact input tensors**

This isolates whether the remaining mismatch comes from tensor construction, tokenizer/model instance differences, or checkpoint loading differences.


In [26]:
import numpy as np
import torch

# Use separate model objects to avoid variable reuse confusion.
st_model = NotebookFunctions.initialize_model(
    "../models/NER_mudel_v2/", unique_labels, cleanup=False
)
hf_bundle_diag = initialize_model("../models/NER_mudel_v2/", cleanup=False)
hf_model_diag = hf_bundle_diag["model"]
hf_tokenizer_diag = CamembertTokenizer.from_pretrained(
    "../models/NER_mudel_v2/", use_fast=False
)
hf_label2id_diag = hf_bundle_diag["label2id"]

# Keep only columns used by both pipelines.
diag_df = ud_test_df[["sentence_id", "words", "labels"]].copy()

# --- A) Build tensors from both pipelines ---
st_eval_dataset = st_model.load_and_cache_examples(
    diag_df, evaluate=True, no_cache=True
)
st_input_ids = st_eval_dataset.tensors[0].cpu().numpy()
st_attention = st_eval_dataset.tensors[1].cpu().numpy()
st_label_ids = st_eval_dataset.tensors[3].cpu().numpy()

diag_sentences = _group_sentences_from_df(diag_df)
hf_encodings_diag, hf_labels_diag = prepare_token_classification_data(
    tokenizer=hf_tokenizer_diag,
    sentences=diag_sentences,
    label2id=hf_label2id_diag,
    max_length=128,
    ignore_placeholders=False,
    pad_token_label_id=-100,
)
hf_input_ids = np.asarray(
    [enc["input_ids"] for enc in hf_encodings_diag], dtype=np.int64
)
hf_attention = np.asarray(
    [enc["attention_mask"] for enc in hf_encodings_diag], dtype=np.int64
)
hf_label_ids = np.asarray(hf_labels_diag, dtype=np.int64)

print(
    "=== Tensor parity check (SimpleTransformers dataset vs custom preprocessing) ==="
)
print("ST shapes:", st_input_ids.shape, st_attention.shape, st_label_ids.shape)
print("HF shapes:", hf_input_ids.shape, hf_attention.shape, hf_label_ids.shape)

same_shape = (
    st_input_ids.shape == hf_input_ids.shape
    and st_attention.shape == hf_attention.shape
    and st_label_ids.shape == hf_label_ids.shape
)
print("Same shapes:", same_shape)

if same_shape:
    input_tensor_equal = np.array_equal(st_input_ids, hf_input_ids)
    attention_tensor_equal = np.array_equal(st_attention, hf_attention)
    labels_tensor_equal = np.array_equal(st_label_ids, hf_label_ids)

    print("input_ids equal:", input_tensor_equal)
    print("attention_mask equal:", attention_tensor_equal)
    print("label_ids equal:", labels_tensor_equal)

    if not input_tensor_equal:
        # Find first sentence with any token id mismatch.
        row_idx = int(np.where((st_input_ids != hf_input_ids).any(axis=1))[0][0])
        print(f"First input_ids mismatch sentence index: {row_idx}")
        st_tokens = st_model.tokenizer.convert_ids_to_tokens(
            st_input_ids[row_idx].tolist()
        )
        hf_tokens = hf_tokenizer_diag.convert_ids_to_tokens(
            hf_input_ids[row_idx].tolist()
        )
        first_pos = int(np.where(st_input_ids[row_idx] != hf_input_ids[row_idx])[0][0])
        start = max(0, first_pos - 6)
        end = min(len(st_tokens), first_pos + 7)
        print("ST tokens window:", st_tokens[start:end])
        print("HF tokens window:", hf_tokens[start:end])

else:
    print("Shapes differ: compare dataset ordering/sentence grouping first.")

# --- B) Compare raw model outputs on identical tensors (ST tensors as common input) ---
device_diag = torch.device("cuda" if torch.cuda.is_available() else "cpu")
st_model.model.to(device_diag).eval()
hf_model_diag.to(device_diag).eval()

batch_size_diag = 64
total_active = 0
pred_mismatch = 0
logit_max_abs_diff = 0.0

with torch.no_grad():
    for start_idx in range(0, st_input_ids.shape[0], batch_size_diag):
        end_idx = min(start_idx + batch_size_diag, st_input_ids.shape[0])

        batch_ids = torch.tensor(st_input_ids[start_idx:end_idx], device=device_diag)
        batch_attn = torch.tensor(st_attention[start_idx:end_idx], device=device_diag)
        batch_labels = st_label_ids[start_idx:end_idx]

        st_logits = st_model.model(
            input_ids=batch_ids, attention_mask=batch_attn
        ).logits
        hf_logits = hf_model_diag(input_ids=batch_ids, attention_mask=batch_attn).logits

        diff = (st_logits - hf_logits).abs().max().item()
        if diff > logit_max_abs_diff:
            logit_max_abs_diff = diff

        st_pred = st_logits.argmax(dim=-1).cpu().numpy()
        hf_pred = hf_logits.argmax(dim=-1).cpu().numpy()

        active_mask = batch_labels != -100
        total_active += int(active_mask.sum())
        pred_mismatch += int((st_pred[active_mask] != hf_pred[active_mask]).sum())

print("\n=== Model output parity on identical tensors ===")
print(f"Active tokens compared: {total_active}")
print(
    f"Prediction mismatches on same tensors: {pred_mismatch} "
    f"({(pred_mismatch / total_active) if total_active else 0.0:.2%})"
)
print(f"Max absolute logits difference: {logit_max_abs_diff:.6f}")

if pred_mismatch > 0:
    print(
        "Non-zero mismatch on identical tensors suggests model/checkpoint loading differences (or runtime precision differences)."
    )
else:
    print(
        "Predictions match on identical tensors. Remaining mismatch likely comes from differences in data fed into each eval call."
    )

100%|██████████| 3/3 [00:09<00:00,  3.19s/it]


=== Tensor parity check (SimpleTransformers dataset vs custom preprocessing) ===
ST shapes: (1198, 128) (1198, 128) (1198, 128)
HF shapes: (1198, 128) (1198, 128) (1198, 128)
Same shapes: True
input_ids equal: True
attention_mask equal: True
label_ids equal: True

=== Model output parity on identical tensors ===
Active tokens compared: 47534
Prediction mismatches on same tensors: 0 (0.00%)
Max absolute logits difference: 0.000000
Predictions match on identical tensors. Remaining mismatch likely comes from differences in data fed into each eval call.


In [ ]:
# Re-initialise with SimpleTransformers-compatible tokenizer selection (CamemBERT => slow tokenizer).
hf_bundle_diag2 = initialize_model("../models/NER_mudel_v2/", use_fast_tokenizer=False)
hf_model_diag2 = hf_bundle_diag2["model"]
hf_tokenizer_diag2 = hf_bundle_diag2["tokenizer"]
hf_label2id_diag2 = hf_bundle_diag2["label2id"]

print("HF tokenizer class:", hf_tokenizer_diag2.__class__.__name__)
print("HF tokenizer is_fast:", getattr(hf_tokenizer_diag2, "is_fast", None))

# Quick spot-check on sentence 0 token window used earlier.
diag_sent_0_words, diag_sent_0_labels = sentences[0]
st_feature_0 = build_st_feature_for_sentence(
    words=diag_sent_0_words,
    labels=diag_sent_0_labels,
    tokenizer_obj=st_model.tokenizer,
    label2id_map=hf_label2id_diag2,
    max_seq_length=128,
    pad_token_label_id=-100,
)
hf_enc_0, hf_lab_0 = prepare_token_classification_data(
    tokenizer=hf_tokenizer_diag2,
    sentences=[(diag_sent_0_words, diag_sent_0_labels)],
    label2id=hf_label2id_diag2,
    max_length=128,
    ignore_placeholders=False,
    pad_token_label_id=-100,
)

st_tokens_0 = st_model.tokenizer.convert_ids_to_tokens(st_feature_0["input_ids"])
hf_tokens_0 = hf_tokenizer_diag2.convert_ids_to_tokens(hf_enc_0[0]["input_ids"])
print("ST window:", st_tokens_0[:8])
print("HF window:", hf_tokens_0[:8])
print("Window equal:", st_tokens_0[:8] == hf_tokens_0[:8])

HF tokenizer class: CamembertTokenizer
HF tokenizer is_fast: False
ST window: ['<s>', '▁Palju', '▁olulisi', '▁kompon', 'ente', '▁,', '▁nagu', '▁liha']
HF window: ['<s>', '▁Palju', '▁olulisi', '▁kompon', 'ente', '▁,', '▁nagu', '▁liha']
Window equal: True
